Trying to extract the most prominent k-mers from the reactive, bystander, self tolerant, and pre-immune datasets to compare what is the most prevalent marker of a tcr from that item

In [88]:
import polars as pl
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

In [89]:
data_dir_kb = Path("../Data/20250910 Comparison 2/Kb")
data_dir_kd = Path("../Data/20250910 Comparison 2/Kd")
output_dir = Path("Comparison2_v2")
output_dir.mkdir(exist_ok=True)

In [90]:
kb_reactive = pl.concat([pl.read_csv(data_dir_kb / f"20250910 B10BR PD1hi{x} LL TCR Repertoire.csv") for x in 'ABCDE'])

kb_selfTolerant = pl.concat([pl.read_csv(data_dir_kb / "20250910 1783 Naive LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 1783 Naive SLO TCR Repertoire.csv")])

kb_bystander = pl.read_csv(data_dir_kb / "20250910 B10BR PD1negD LL TCR Repertoire.csv")

kb_preImmune = pl.concat([pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR pre-immmune TCR Repertoire.csv")])

In [91]:
kd_reactive = pl.concat([pl.read_csv(data_dir_kd / f"20251120 BL6 PD1hi{x} LL TCR Repertoire.csv") for x in 'ABC'])

kd_selfTolerant = pl.concat([pl.read_csv(data_dir_kd / "20250910 B6Kd Naive LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 B6Kd Naive SLO TCR Repertoire.csv")])

kd_bystander = pl.concat([pl.read_csv(data_dir_kd / "20250910 BL6 PD1negA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 PD1negB LL TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 BL6 PD1negC LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20251120 BL6 PD1negD LL TCR Repertoire.csv").drop('')])

kd_preImmune = pl.concat([pl.read_csv(data_dir_kd / "20250910 BL6 NaiveA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 NaiveA SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 BL6 NaiveB LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 NaiveB SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 C57BL6 Pre-Immune SLO TCR Repertoire.csv")])

In [92]:
def extract_kmers(df, col_name, out_alias, k):
    """Return all k-mers from an amino-acid sequence.
    Args:
        df - pl.DataFrame
        col_name - str,
        out_alias - str,
        k - int
    Returns:
        pl.DataFrame
    """
    return(
        df.select(
            pl.concat_list([
                pl.col(col_name).str.slice(i, k)
                for i in range(df[col_name].str.len_chars().max() - k + 1)
            ]).alias(f'{k}-mer')
        )
        .explode(f'{k}-mer', empty_as_null=True)
        .filter(pl.col(f'{k}-mer').str.len_chars() == k)
        .group_by(f'{k}-mer')
        .len(name=out_alias)
    )

In [93]:
def counts(df_list, col_name, aliases, k):
    """Merge all the counts into one table
    Args:
        df_list - list
        col_name - str
        aliases - list
        k - int
    Returns:
        pl.DataFrame"""
    basis = extract_kmers(df_list[0], col_name, aliases[0], k)
    for df, alias in zip(df_list[1:], aliases[1:]):
        counts = extract_kmers(df, col_name, alias, k)
        basis = basis.join(counts, on=f'{k}-mer', how='full', coalesce=True)
    return basis.fill_null(0)

In [94]:
alpha = 'CDR3a_aa'
beta = 'CDR3b_aa'

Counting all 3-mers across the different datasets

In [95]:
kb_cdr3a_counts_df = counts(
    [kb_reactive, kb_selfTolerant, kb_preImmune], alpha,
    ['kb_reactive_cdr3a', 'kb_selfTolerant_cdr3a', 'kb_preImmune_cdr3a'], 3
)

kb_cdr3b_counts_df = counts(
    [kb_reactive, kb_selfTolerant, kb_preImmune], alpha,
    ['kb_reactive_cdr3b', 'kb_selfTolerant_cdr3b', 'kb_preImmune_cdr3b'], 3
)

kd_cdr3a_counts_df = counts(
    [kd_reactive, kd_selfTolerant, kd_preImmune], alpha,
    ['kd_reactive_cdr3a', 'kd_selfTolerant_cdr3a', 'kd_preImmune_cdr3a'], 3
)

kd_cdr3b_counts_df = counts(
    [kd_reactive, kd_selfTolerant, kd_preImmune], alpha,
    ['kd_reactive_cdr3b', 'kd_selfTolerant_cdr3b', 'kd_preImmune_cdr3b'], 3
)

kb_cdr3a_counts_df.write_csv(output_dir / 'kb_cdr3a_3mer_counts.csv')
kb_cdr3b_counts_df.write_csv(output_dir / 'kb_cdr3b_3mer_counts.csv')
kd_cdr3a_counts_df.write_csv(output_dir / 'kd_cdr3a_3mer_counts.csv')
kd_cdr3b_counts_df.write_csv(output_dir / 'kd_cdr3b_3mer_counts.csv')

Z scores of the 3-mers, relative to the row average - i.e. does this 3-mer appear more or less often than the average amount

Fold change of the 3-mer relative to the pre-immune dataset, expecting the key 3-mers to be up-regulated in the reactive and down-regulated in the selfTolerant

In [96]:
def freq_and_z(df, k):
    """Computes the frequencies of all the k-mers and then calculates the relative z-scores to compare between
        reactive, self tolerant, and pre immune datasets
    Args:
        df -> pl.DataFrame
    Returns:
        tuple[pl.DataFrame, pl.DataFrame]"""
    cols = [c for c in df.columns if c != f'{k}-mer']

    freq_df = df.with_columns([pl.col(c) / pl.col(c).sum() for c in cols])
    row_mean = pl.mean_horizontal(cols)
    variance_sum = pl.sum_horizontal([(pl.col(c) - row_mean) ** 2 for c in cols])
    row_std = (variance_sum / (len(cols) - 1)).sqrt()

    zscore_df = freq_df.with_columns([
        ((pl.col(c) - row_mean) / row_std).alias(c) for c in cols
    ])

    return freq_df, zscore_df

In [97]:
kb_cdr3a_freq_df, kb_cdr3a_z_df = freq_and_z(kb_cdr3a_counts_df, 3)
kb_cdr3b_freq_df, kb_cdr3b_z_df = freq_and_z(kb_cdr3b_counts_df, 3)
kd_cdr3a_freq_df, kd_cdr3a_z_df = freq_and_z(kd_cdr3a_counts_df, 3)
kd_cdr3b_freq_df, kd_cdr3b_z_df = freq_and_z(kd_cdr3b_counts_df, 3)

Making a heatmap to visualise the z score - picked the 3-mers with the highest variance

In [98]:
datasets = [
    ('kb_cdr3a', kb_cdr3a_freq_df, kb_cdr3a_z_df),
    ('kb_cdr3b', kb_cdr3b_freq_df, kb_cdr3b_z_df),
    ('kd_cdr3a', kd_cdr3a_freq_df, kd_cdr3a_z_df),
    ('kd_cdr3b', kd_cdr3b_freq_df, kd_cdr3b_z_df)
]

for name, freq_df, z_df in datasets:
    cols = [c for c in freq_df.columns if c != '3-mer']

    most_variable = (freq_df.select([
        pl.col('3-mer'),
        (pl.sum_horizontal([pl.col(c) - pl.mean_horizontal(cols) ** 2 for c in cols]) / (len(cols) - 1)).alias('var')
        ])
        .sort('var', descending=True)
        .head(500)
        .join(z_df, on='3-mer', how='inner')
        .drop('var')
        )
    
    labels = most_variable['3-mer'].to_numpy()
    numbers = most_variable.select(cols).to_numpy()
    
    sns_df = pd.DataFrame(numbers, columns=cols, index=labels)

    sns.clustermap(sns_df, cmap='vlag', center=0, figsize=(10,70), yticklabels=True)
    plt.savefig(output_dir / f'{name}_zscore_heatmap.png')
    plt.close()